# SoloLab usage examples

Worked examples for reading, processing and plotting STIX / RPW-HFR / RPW-TNR / EPD data directly through `sololab`'s functions - the same functions both the desktop and web apps call under the hood.

> **Note:** `sololab_tutorial.ipynb` (also in this repo) is outdated and no longer works with the current codebase (see `README.md`). This notebook replaces it.

Run `python test_installation.py` first if you haven't already, to confirm your environment has everything this notebook needs.

## Sample data

STIX and RPW examples need a local file to read (they don't auto-download). If you don't have any yet:
- STIX: [STIX data center](https://datacenter.stix.i4ds.net/) - a spectrogram FITS file (filename contains `spec`).
- RPW: [SOAR](https://soar.esac.esa.int/soar/) - search for `RPW-HFR-SURV` or `RPW-TNR-SURV`, L2 or L3, CDF format.

Drop them anywhere under `data/` (already `.gitignore`d) and update the paths in the next cell. EPD needs no local file - it downloads automatically via `solo_epd_loader`.

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

from sololab import stix_read, rpw_read, quicklooks
from solo_epd_loader import epd_load

%matplotlib inline

In [ ]:
DATA_DIR = Path("data")

# Auto-detect a sample file of each type if present; otherwise these stay None and the
# corresponding section below just explains what to do instead of erroring out.
stix_file = next(iter(sorted(DATA_DIR.glob("*stix*.fits"))), None)
hfr_file = next(iter(sorted(DATA_DIR.glob("*rpw-hfr*.cdf"))), None)
tnr_file = next(iter(sorted(DATA_DIR.glob("*rpw-tnr*.cdf"))), None)

print("STIX file:", stix_file)
print("RPW-HFR file:", hfr_file)
print("RPW-TNR file:", tnr_file)

## 1. STIX: reading and plotting an X-ray spectrogram

`stix_read.stix_create_counts(path)` reads a STIX FITS file (spectrogram or L1 imaging - auto-detected from the filename) into a `dict` with `time`, `counts_per_sec`, `energy_bins` (an `astropy.table.Table`), and `mean_energy`.

In [ ]:
if stix_file is not None:
    stix_counts = stix_read.stix_create_counts(str(stix_file))
    print("Keys:", list(stix_counts.keys()))
    print("Time range:", stix_counts["time"].min(), "to", stix_counts["time"].max())
    print("Counts array shape (time, energy channels):", stix_counts["counts_per_sec"].shape)
else:
    print("No STIX sample file found under data/ - see the intro cell for where to get one.")

In [ ]:
if stix_file is not None:
    fig, ax = plt.subplots(figsize=(10, 4))
    quicklooks.stix_plot_spectrogram(stix_counts, ax=ax, x_axis=True, logscale=True)
    ax.set_title(stix_file.name)
    plt.show()

### Background subtraction

`stix_read.stix_remove_bkg_counts` subtracts a background estimated either from a separate background FITS file (`pathbkg=...`), a quiet time range within the same file (`stix_bkg_range=(start, end)`, format `'%d-%b-%Y %H:%M:%S'`), or both at once. This example uses a time range - adjust it to a genuinely quiet interval within your file's time span.

In [ ]:
if stix_file is not None:
    t0, t1 = stix_counts["time"].min(), stix_counts["time"].max()
    quiet_start = t0.strftime("%d-%b-%Y %H:%M:%S")
    quiet_end = (t0 + (t1 - t0) * 0.05).strftime("%d-%b-%Y %H:%M:%S")  # first 5% of the file, as an example

    stix_counts_nobkg = stix_read.stix_remove_bkg_counts(
        str(stix_file), stix_bkg_range=(quiet_start, quiet_end), bkg_poll_function="mean",
    )
    fig, ax = plt.subplots(figsize=(10, 4))
    quicklooks.stix_plot_spectrogram(stix_counts_nobkg, ax=ax, x_axis=True, logscale=True)
    ax.set_title(f"{stix_file.name} (background subtracted)")
    plt.show()

## 2. RPW-HFR / RPW-TNR: reading and plotting radio spectrograms

`rpw_read.rpw_get_data(path)` auto-detects HFR vs. TNR and L2 vs. L3 from the filename and reads the raw CDF. `rpw_read.rpw_create_PSD(data, which_freqs="non_zero")` turns that into a power-spectral-density dict (`time`, `frequency`, `v`, ...) ready to plot.

In [ ]:
if hfr_file is not None:
    hfr_raw = rpw_read.rpw_get_data(str(hfr_file))
    hfr_psd = rpw_read.rpw_create_PSD(hfr_raw, which_freqs="non_zero")
    print("HFR PSD shape (frequency, time):", hfr_psd["v"].shape)

    fig, ax = plt.subplots(figsize=(10, 4))
    quicklooks.rpw_plot_psd(hfr_psd, ax=ax, frequency_range=[0, 17000], rpw_cbar_units="wmhz")
    ax.set_title(hfr_file.name)
    plt.show()
else:
    print("No RPW-HFR sample file found under data/ - see the intro cell for where to get one.")

In [ ]:
if tnr_file is not None:
    tnr_raw = rpw_read.rpw_get_data(str(tnr_file))
    tnr_psd = rpw_read.rpw_create_PSD(tnr_raw, which_freqs="non_zero")

    fig, ax = plt.subplots(figsize=(10, 4))
    quicklooks.rpw_plot_psd(tnr_psd, ax=ax, rpw_cbar_units="wmhz")
    ax.invert_yaxis()  # TNR is conventionally shown with frequency decreasing upward
    ax.set_title(tnr_file.name)
    plt.show()
else:
    print("No RPW-TNR sample file found under data/ - see the intro cell for where to get one.")

### Time-profile curves at fixed frequencies

`rpw_plot_curves` plots the PSD at one or more chosen frequencies (nearest available channel) as time series, instead of the full spectrogram. The default `bias_multiplier=1` vertically offsets each curve into a "waterfall" so overlapping curves stay legible, labelling each with a text annotation. Pass `bias_multiplier=None` instead for plain overlaid curves with a legend.

In [ ]:
if hfr_file is not None:
    fig, ax = plt.subplots(figsize=(10, 3))
    quicklooks.rpw_plot_curves(hfr_psd, ax=ax, freqs=[500, 3500, 13000], ylogscale=True)
    plt.show()

## 3. EPD: downloading and plotting energetic-particle flux

Unlike STIX/RPW, EPD data isn't read from a local file you provide - `solo_epd_loader.epd_load(..., autodownload=True)` fetches it from the SOAR archive for you (and caches it locally under `path=` for next time). This cell needs network access; it downloads a few MB.

In [ ]:
EPD_DATE = 20221225  # YYYYMMDD, as an int - pick a date your STIX/RPW files also cover for section 4

df_protons_ept, df_electrons_ept, energies_ept = epd_load(
    sensor="ept", level="l2", startdate=EPD_DATE, enddate=EPD_DATE,
    viewing="sun", path="data/epd", autodownload=True,
)
print("Electron flux columns:", list(df_electrons_ept.columns.get_level_values(0).unique()))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
quicklooks.plot_ept_data(
    df_electrons_ept, energies_ept, ax, particle="Electron",
    channels=[2, 6, 14, 18, 26], resample="1min",
)
plt.show()

## 4. Combined plot: STIX + RPW-HFR + RPW-TNR + EPD together

`quicklooks.quicklook_plot` stacks any subset of the four instruments into one figure with a shared time axis - this is what both the desktop and web apps' "Combined Plot" feature calls. `display` controls which panels appear and in what order; `stix_mode`/`rpw_mode` choose `'spec'` (spectrogram), `'curve'` (time profiles), or `'overlay'` (both, on twin axes) per instrument family.

Only runs if all four datasets from the sections above were loaded - comment out whichever instruments you skipped, and drop them from `display` too.

In [ ]:
have_all = stix_file is not None and hfr_file is not None and tnr_file is not None
if have_all:
    fig = quicklooks.quicklook_plot(
        stix_counts=stix_counts,
        hfr_psd=hfr_psd,
        tnr_psd=tnr_psd,
        epd_data=df_electrons_ept,
        epd_energies=energies_ept,
        display=["stix", "hfr", "tnr", "epd"],
        stix_mode="curve",       # per-energy-bin count-rate curves
        rpw_mode="spec",         # spectrograms for both HFR and TNR
        epd_particle="Electron",
        figsize=(12, 10),
    )
    plt.show()
else:
    print("Need STIX + RPW-HFR + RPW-TNR sample files for this example - see the intro cell.")

## See also

- **Desktop app** (`sololab_app.py`) and **web app** (`sololab/dash_app/`) wrap all of the above in a GUI, plus background-subtraction UI, frequency/channel selection, and saving/loading a "data pack" of everything you've loaded in a session.
- **Frequency Drift Rate Analysis** (`sololab/freq_drift.py`) and **electron powerlaw estimation** (`sololab/electron_powerlaw.py`) implement the "Estimations and fits" capability mentioned in `README.md`, but aren't wired into either GUI yet - call their functions directly if you need them. Not covered here: they weren't exercised in the same session this notebook's examples were verified against, so check their docstrings/source before relying on them.